# Auxiliar 1 - Series estacionarias y no estacionarias



## Serie temporal de pasajes aéreos

Planteamos la descomposición y el análisis de la serie temporal de pasajes aéreos. Comenzaremos cargando la base de datos y agregando un índice de tiempo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

AP = pd.read_csv('Data/AirPassengers.csv')
AP.head()


In [ ]:
AP.set_index(pd.to_datetime(AP['date']),inplace=True)
AP['Interval'] = np.float32((AP.index - AP.index[0]).days)

Realizamos una representación para recordar como varía la variable AP (pasajes aéreos) con el tiempo.

In [ ]:
AP['AP'].plot()
plt.ylabel('Pasajes aéreos (miles)')

Notamos una tendencia positiva para la variable AP y un claro comportamiento estacional. Además la oscilación intranual se amplifica con la tendencia por lo que sugeriría un modelo de descomposición multiplicativo.


### Análisis de los momentos de la distribución 



Observamos que la serie presenta tendencia y efectos estacionales. Así, la serie no es estacionaria. Esto quiere decir que los momentos de la distribución, como la media o la varianza, cambian con el tiempo. La propuesta es mostrar el valor que toman estos estadísticos al ampliar la longitud de la serie. 

In [ ]:
data_t = np.arange(1, AP['AP'].size)
mean_t = np.array([])
var_t = np.array([])
for i in data_t:
    mean_t = np.append(mean_t, [np.mean(AP['AP'][:i])])
    var_t = np.append(var_t, [np.var(AP['AP'][:i])])

fig, ax = plt.subplots(1,2, figsize=(8,4))

ax[0].plot(AP.index[1:], mean_t)
ax[0].set_xlabel('Fecha')
ax[0].set_ylabel('Media')

ax[1].plot(AP.index[1:],var_t)
ax[1].set_xlabel('Fecha')
ax[1].set_ylabel('Varianza')

plt.tight_layout()


Claramente, se observa que la serie no es estacionaria: los valoes de los primeros momentos de la distribución cambian con el tiempo. Asimismo, la autocorrelación pasa a depender, no solo del desplazamiento temporal (lag), sino también del instante en que se inicia el análisis. Esto es por definición de la función de autocorrelación en la que interviene el valor medio.  

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(1,2, figsize=(8,4))

quarter_size = AP['AP'].size // 4

plot_acf(AP['AP'][:quarter_size], lags=20, bartlett_confint=False, ax=ax[0])
ax[0].set_xlabel('Lags')
ax[0].set_title('Autocorrelación 1er cuarto')

plot_acf(AP['AP'][3*quarter_size:], lags=20, bartlett_confint=False, ax=ax[1])
ax[1].set_xlabel('Lags')
ax[1].set_title('Autocorrelación 4to cuarto')


plt.tight_layout()

### Análisis en el tiempo para la serie diferencianda

Un abordaje al problema de la series no estacionarias es plantear la diferencia entre 1 o más pasos de tiempo. Aquí, dado que la serie presenta un efecto estacional de 12 meses, podríamos diferenciar la serie tomando dicha separación en pasos de tiempo. Asimismo, puede anular los cambios por tendencia. Por otro lado, como la varianza se amplifica con la tendencia, en estos casos una transformación logarítmica estabiliza la curva. 

Finalmente, podemos tomar diferentes combinaciones de pasos para la diferenciación y transformaciónes en los datos. En este caso, nos inclinamos por una diferenciación a un paso y una transformación logarítmica. Vamos a buscar de confirmar que la serie diferenciada es estacionaria analizando los gráficos ya vistos como, también, con el test de Dickey-Fuller.



In [ ]:
DIFF_STEPS = 1
AP_DIFF = np.log(AP['AP']).diff(DIFF_STEPS)[DIFF_STEPS:]
# Arranca en DIFF_STEPS para quitar los NaN 

data_t = np.arange(1, AP_DIFF.size)
mean_t = np.array([])
var_t = np.array([])
for i in data_t:
    mean_t = np.append(mean_t, [np.mean(AP_DIFF[:i])])
    var_t = np.append(var_t, [np.var(AP_DIFF[:i])])

fig, ax = plt.subplots(1,2, figsize=(8,4))

ax[0].plot(AP.index[DIFF_STEPS+1:], mean_t)
ax[0].set_xlabel('Fecha')
ax[0].set_ylabel('Media')

ax[1].plot(AP.index[DIFF_STEPS+1:],var_t)
ax[1].set_xlabel('Fecha')
ax[1].set_ylabel('Varianza')

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(8,4))

quarter_size = AP_DIFF.size // 4

plot_acf(AP_DIFF[:quarter_size], lags=20, bartlett_confint=False, ax=ax[0])
ax[0].set_xlabel('Lags')
ax[0].set_title('Autocorrelación 1er cuarto')

plot_acf(AP_DIFF[3*quarter_size:], lags=20, bartlett_confint=False, ax=ax[1])
ax[1].set_xlabel('Lags')
ax[1].set_title('Autocorrelación 4to cuarto')


plt.tight_layout()

#### Chequeo con test de Dickey Fuller

In [ ]:
from statsmodels.tsa.stattools import adfuller
ADF_result = adfuller(AP_DIFF)
print(f'ADF Statistic: {ADF_result[0]:.3f}')
print(f'p-value: {ADF_result[1]:.3e}')

print("Critical Values:")
for key, value in ADF_result[4].items():
    print(f"   {key}: {value:.3f}")

Ver también [ANALYSIS OF THE AIRPASSENGERS DATASET](https://rpubs.com/Sandra_N_Busieka/1209533).

### Descomposición de la serie temporal de AP

Introducimos la función `seasonal_decompose` del paquete `statsmodels.tsa.seasonal` que realiza la descomposición de la serie en tendencia, efectos estacionales y residuo.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
period = 12 #setting the period for decomposition
# Apply seasonal_decompose
result_sd = seasonal_decompose(AP['AP'], model='multiplicative', period=period, extrapolate_trend=0)
# Plot the results
# plot_components(result_sd)
# or
result_sd.plot();
#plt.gca().set_xticks(np.arange(0,3000,365),np.arange(0,9))
plt.gca().set_xlabel('Year')